# Formula scores, positions and exact next-day returns

SYNTHETIC METHOD DEMONSTRATION. Adapted from reviewed R1/R2 methods. No historical market or platform evidence is embedded.

Run the actual frozen dual evaluator on synthetic data with a declared teaching formula.

In [ ]:
from alpha_discovery.synthetic import make_panel
from alpha_discovery.expressions import Interpreter, parse
from alpha_discovery.local_first_evaluator import DualEvaluator
import numpy as np
panel, *_ = make_panel()
signal = Interpreter(panel).evaluate(parse('rank(ts_mean(returns,5))')).to_numpy()
evaluator = DualEvaluator(panel)
result, daily, books = evaluator.compute(signal, {'candidate_id': 'SYNTHETIC_NOTEBOOK'}, retain_weights=True)
print({'mode': 'SYNTHETIC', 'fixed_direction': result['fixed_direction'], 'rows': len(daily)})

Inspect actual weights and preserve unknown daily P&L rather than filling it with zero.

In [ ]:
w = books['wq']
defined = np.isfinite(w).all(axis=1)
assert np.allclose(np.abs(w[defined]).sum(axis=1), 2)
assert np.allclose(w[defined].sum(axis=1), 0, atol=1e-10)
assert np.max(np.abs(w[defined])) <= .08 + 1e-12
assert daily.loc[~daily.registered_endpoint_valid,'wq_gross_fixed'].isna().all()
result['evaluations']['wq']['periods']['development_2022_2024']